In [46]:
import numpy as np
from scipy.signal import place_poles
import numpy.typing as npt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, NonlinearConstraint
N = 6
observable = False

def init_system(N:int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    A = np.eye(N, k=-1)
    A[:, -1] = np.random.randint(-N*2, -1, N)
    B = np.random.randint(-N, N, N).reshape(N,1)
    C = np.eye(1,N, k=N-1)
    return A, B, C

def get_observ_params(A: np.ndarray, C: np.ndarray, N :int) -> np.ndarray:
    desired_poles = np.random.choice(np.arange(-15, -6, 1), N, replace=False)
    res = place_poles(A.T, C.T, desired_poles)
    return res.gain_matrix

def f(N) -> npt.NDArray:
        return np.round(
            
            np.random.randint(-N*3, N*3,N)
            + np.random.random(N),
            3,
        ).reshape(1,N)
# def f(N) -> npt.NDArray:
#         l_true = -np.random.exponential(2, N-2)
#         g_coeffs = np.random.normal(0, 3, N-2) 
#         t = np.arange(N-1)
#         basis_matrix = np.array([lt**N for lt in l_true]).T
#         f_true = basis_matrix @ g_coeffs
#         f_noisy = f_true + np.random.normal(0, 1, N)
#         return f_noisy.reshape(1,N)

A, B, C = init_system(N)
while not observable:
    Obsv = np.vstack([ C @ np.linalg.matrix_power(A, i) for i in range(N)])
    if np.linalg.matrix_rank(Obsv) < A.shape[0]:
        print(Obsv)
        print(np.linalg.matrix_rank(Obsv))
        A, B, C = init_system(N)
    else:
        observable = True

stable = False
F = f(N)
k_sigma = 1.5
while not stable:
    A_cl = A - k_sigma * B @ F
    if np.all(np.linalg.eigvals(A_cl) < 0):
        stable = True
    else:
        # print(F)
        F = f(N)    
L = get_observ_params(A, C, N)
T = 100

In [47]:
A, B, C, F, L, k_sigma, N,

(array([[  0.,   0.,   0.,   0.,   0.,  -6.],
        [  1.,   0.,   0.,   0.,   0.,  -7.],
        [  0.,   1.,   0.,   0.,   0., -11.],
        [  0.,   0.,   1.,   0.,   0., -12.],
        [  0.,   0.,   0.,   1.,   0.,  -5.],
        [  0.,   0.,   0.,   0.,   1., -12.]]),
 array([[ 3],
        [-4],
        [-4],
        [-4],
        [-4],
        [ 1]]),
 array([[0., 0., 0., 0., 0., 1.]]),
 array([[17.705, -5.906,  0.074, -7.867, -6.024, 13.086]]),
 array([[1.10073001e+06, 6.72161005e+05, 1.68707001e+05, 2.22690001e+04,
         1.62800001e+03, 5.10000001e+01]]),
 1.5,
 6)

In [48]:
A, B, C, F, L, k_sigma, N = (np.array([[ 0.,  0.,  0.,  0.,  0., -2.],
        [ 1.,  0.,  0.,  0.,  0., -4.],
        [ 0.,  1.,  0.,  0.,  0., -6.],
        [ 0.,  0.,  1.,  0.,  0., -8.],
        [ 0.,  0.,  0.,  1.,  0., -6.],
        [ 0.,  0.,  0.,  0.,  1., -3.]]),
 np.array([[ 4],
        [ 4],
        [ 2],
        [-2],
        [ 5],
        [-1]]),
 np.array([[0., 0., 0., 0., 0., 1.]]),
 np.array([[ 13.422,  12.56 ,  -1.103, -14.334,   7.709, -16.452]]),
 np.array([[1.03487800e+06, 6.34923997e+05, 1.60437999e+05, 2.13719999e+04,
         1.57900000e+03, 5.89999999e+01]]),
 1.5,
 6)

In [49]:
def wandermond_lambdas_matrix(lambdas: npt.NDArray, n: int) -> npt.NDArray:
    return np.vstack([lambdas**i for i in range(n)]).T
    
def wandermond_matrix(gammas: npt.NDArray, lambdas: npt.NDArray, n: int) -> npt.NDArray:
    return np.vstack([gammas * lambdas**i for i in range(n)]).T

def approximation_elemnet(
    gammas: npt.NDArray, lambdas: npt.NDArray, n: int
) -> npt.NDArray:
    return np.sum(wandermond_matrix(gammas, lambdas, n), axis=0)


def functional_J(
    f: npt.NDArray, gammas: npt.NDArray, lambdas: npt.NDArray, n: int
) -> float:
    diff = (f - approximation_elemnet(gammas, lambdas, n)).flatten()
    return diff @ diff
def approximation_F(F: npt.NDArray, K:int, N: int, lambdas: list[float] = None, gammas: list[float] = None, lambda_coef: float = 1.0) -> npt.NDArray:
    lambdas = np.array([-float(i)*lambda_coef  for i in range(1, K + 1)])
    lambda_m  = wandermond_lambdas_matrix(lambdas, N)
    gramm = lambda_m @ lambda_m.T
    b_vec = lambda_m @ F.flatten()
    gammas = np.linalg.solve(gramm, b_vec)
    return gammas, lambdas



In [50]:
def run_system(A: np.ndarray, B: np.ndarray, F: np.ndarray, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        u = -k_sigma * F @ x
        x_dot = A @ x + B @ u
        return x_dot.flatten()
    
    x0 = np.ones(N)
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    sigma_ideal = (F @ sol.y).flatten()
    u_ideal = -k_sigma * sigma_ideal.flatten()
    return  u_ideal, sigma_ideal

u_ideal, sigma_ideal = run_system(A, B, F, N, 0.1, T)
# u_ideal, sigma_ideal

In [51]:
def run_full_observe(A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, F: np.ndarray, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        x_true, x_observe = x[:N], x[N:]
        u = -k_sigma * F @ x_observe
        dx_true = A @ x_true + B @ u
        dx_observe =  A @ x_observe + B @ u + (L * (C @ x_true - C @ x_observe)).flatten()
        return np.concatenate([dx_true, dx_observe]).flatten()
    
    x0 = np.concatenate([np.ones(N), np.zeros(N)])
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    x_true, x_observe = sol.y[:N], sol.y[N:] 
    sigma_full = (F @ x_observe).flatten()   
    u_full = -k_sigma * sigma_full.flatten()
    
    return  u_full, sigma_full

u_full, sigma_full = run_full_observe(A, B, C, L, F, N, 0.1, T)
# u_full, sigma_full

In [52]:
g_lambdas, g_gammas = None, None


def run_func_observe(A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, F: np.ndarray,K: int, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    global g_lambdas, g_gammas
    gammas, lambdas = approximation_F(F, K, N, None, None, 0.2)
    g_lambdas, g_gammas = lambdas.tolist(), gammas.tolist()
    gamma_0 = (F[C==1] - gammas@(lambdas**(N-1))).item()
    print(f"MinJ: {functional_J(F, gammas, lambdas, N)}")
    print(lambdas, gammas, gamma_0)
    LAMBDAS = wandermond_lambdas_matrix(lambdas, N)
    H = (LAMBDAS@B).flatten()
    # print(H)
    G = np.concatenate([-pow_l @ (np.eye(N) * l - A)@C.T for pow_l , l in zip(LAMBDAS, lambdas)])
    # print(G)
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        x_true, sigma = x[:N], x[N:]
        sigma_observe = gammas@sigma +gamma_0*C@x_true
        u = -k_sigma * sigma_observe
        # print(A @ x_true)
        # print(B * u)
        dx_true = A @ x_true + (B * u).flatten()
        # print(lambdas*  sigma_observe)
        # print(H*u)
        # print(G*(C @ x_true))
        dsigma_observe = lambdas* sigma + H*u + G*(C @ x_true)
        # print(lambdas @ (F @ x_observe))
        # print(sigma)
        # print(dx_true)
        # print(dsigma_observe)
        return np.concatenate([dx_true, dsigma_observe]).flatten()
    
    x0 = np.concatenate([np.ones(N), np.zeros(K)])
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    x_true, sigma_observe = sol.y[:N], sol.y[N:]    
    y = (C @ x_true).flatten()  
    sigma_func = gammas@sigma_observe +gamma_0*y
    u_func = -k_sigma * sigma_func
    
    return  u_func, sigma_func

func_observer = {k: run_func_observe(A, B, C, L, F, k, N, 0.1, T) for k in range(1, N)}

MinJ: 758.5598802776617
[-0.2] [10.55822504] -16.448621367986675
MinJ: 689.442344999698
[-0.2 -0.4] [ 47.1387315  -35.05632773] -16.795892401905
MinJ: 106.48486301955107
[-0.2 -0.4 -0.6] [ 271.93795119 -428.96272362  170.71449129] -7.482799303069267
MinJ: 62.65431891610357
[-0.2 -0.4 -0.6 -0.8] [ 142.94726368 -118.40161771  -74.28319518   63.17292264] -2.6944474078337244
MinJ: 57.84851813155476
[-0.2 -0.4 -0.6 -0.8 -1. ] [ 264.31520668 -513.49067707  407.43468721 -198.12125504   53.34245833] -1.521356909116573


In [54]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

dash_styles = ['dash', 'dot', 'dashdot', 'longdash', 'longdashdot', 'solid']
t = np.arange(0, 100, 0.1)
fig = go.Figure(
    layout=dict(
        legend=dict(
            yanchor="top",
            y=-0.2,
            xanchor="left",
            x=0.01,
            orientation="h",
        )
    )
)
fig.update_layout(
    xaxis_title="Время, t",
    yaxis_title="Управление, u",
    plot_bgcolor="white",
)
fig.update_xaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.update_yaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.add_trace(go.Scatter(x=t, y=u_ideal, name='Полный наблюдатель'))
# fig.add_trace(go.Scatter( y=u_full, name='u_full'), row=1, col=1)
for k, (u_func, sigma_func) in func_observer.items():
    dash_с = dash_styles[k % len(dash_styles)]
    fig.add_trace(go.Scatter(x=t, y=u_func, name=f'Наблюдатель при k={k}', line=dict(dash=dash_с)), )
    idx = len(t) // 6 *(k)

    # fig.add_annotation(
    #     x=t[idx],
    #     y=u_func[idx],
    #     text=f'u_func_{k}',
    #     showarrow=False,
    #     yshift=1,
    #     font=dict(size=13),
    #     bgcolor='rgba(255,255,255,0.7)',
    # )

# fig.add_trace(go.Scatter( y=sigma_ideal, name='sigma_ideal'), row=2, col=1)
# fig.add_trace(go.Scatter( y=sigma_full, name='sigma_full'), row=2, col=1)
# for k, (u_func, sigma_func) in func_observer.items():
#     dash_с = dash_styles[k % len(dash_styles)]
#     fig.add_trace(go.Scatter(y=sigma_func, name=f'sigma_func_{k}', line=dict(dash=dash_с)), row=2, col=1)
fig.update_layout(height=600, width=1200)
fig.show()